# Thursday add-on: is that difference real, or just noise?

ASTR 457, Fall 2026 — pulled forward from the original Sep 1/3 slot.

**Live exercise first:** in your terminal, with `astr457` active —

```
conda install -c conda-forge astroml
```

- restart your kernel after it finishes
- this is the same install workflow you'll use all semester when a lab
  needs a new package: run `conda install`, confirm it landed in the right
  env, restart the kernel so the import actually picks it up

*Uses real SDSS data via the `astroML` package (Ivezic, VanderPlas, Gray &
Connolly) — the companion package to our textbook (ICVG).*

## The question

- the Milky Way disk splits (roughly) into a metal-poor, alpha-enhanced
  ("thick disk / old") population and a metal-richer, alpha-normal
  ("thin disk / young") population
- if you split stars by alpha-element abundance, do their metallicities
  ([Fe/H]) actually come from **different distributions** — or could that
  difference in a histogram just be sampling noise?
- **Null hypothesis $H_0$:** the two [Fe/H] samples are drawn from the same
  underlying distribution. We're going to ask the data to talk us out of
  it.

In [ ]:
from astroML.datasets import fetch_sdss_sspp
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

data = fetch_sdss_sspp()

# quality cuts: real data needs them before you trust anything downstream
good = (data['FeHErr'] < 0.3) & (data['alphFeErr'] < 0.3) & (data['SNR'] > 20)
d = data[good]

alpha_poor = d[d['alphFe'] < 0.1]   # thin-disk-like
alpha_rich = d[d['alphFe'] > 0.3]   # thick-disk-like
print(f"alpha-poor: {len(alpha_poor)} stars, alpha-rich: {len(alpha_rich)} stars")

In [ ]:
plt.hist(alpha_poor['FeH'], bins=60, density=True, alpha=0.5, label='alpha-poor')
plt.hist(alpha_rich['FeH'], bins=60, density=True, alpha=0.5, label='alpha-rich')
plt.xlabel('[Fe/H]'); plt.ylabel('density'); plt.legend()
plt.title('Do these look like the same distribution to you?')
plt.show()

**By eye**, obviously different. But "obviously different" is exactly the
kind of judgment call the rest of this course is asking you to back up
quantitatively — same theme as Tuesday's demo, same theme as Lab 01. So:
back it up.

## The Kolmogorov-Smirnov two-sample test

- compares the two samples' empirical cumulative distribution functions
  (ECDFs)
- asks: what's the largest vertical gap between them, $D$?
- and: how often would a gap that big happen by chance, if both samples
  really came from the same distribution?

Let's look at the ECDFs directly before we run the test — the test
statistic is just a number on this plot.

In [ ]:
def ecdf(vals):
    s = np.sort(vals)
    return s, np.arange(1, len(s) + 1) / len(s)

x_poor, y_poor = ecdf(alpha_poor['FeH'])
x_rich, y_rich = ecdf(alpha_rich['FeH'])

plt.plot(x_poor, y_poor, label='alpha-poor ECDF')
plt.plot(x_rich, y_rich, label='alpha-rich ECDF')

# find and mark the largest gap, on a common grid
grid = np.linspace(min(x_poor.min(), x_rich.min()), max(x_poor.max(), x_rich.max()), 2000)
cdf_poor = np.searchsorted(x_poor, grid, side='right') / len(x_poor)
cdf_rich = np.searchsorted(x_rich, grid, side='right') / len(x_rich)
gap = np.abs(cdf_poor - cdf_rich)
i_max = np.argmax(gap)
plt.vlines(grid[i_max], cdf_rich[i_max], cdf_poor[i_max], color='k', linestyle='--',
           label=f'D = {gap[i_max]:.3f}')
plt.xlabel('[Fe/H]'); plt.ylabel('cumulative fraction'); plt.legend()
plt.title('The KS statistic is literally this gap')
plt.show()

In [ ]:
ks_stat, p_value = stats.ks_2samp(alpha_poor['FeH'], alpha_rich['FeH'])
print(f"KS statistic = {ks_stat:.4f}  (matches the gap marked above)")
print(f"p-value = {p_value:.3e}")

## What does that p-value actually mean?

- **Not**: "the probability the null hypothesis is true"
- **Not**: "the probability this result is due to chance," full stop
- **Is**: if the two samples really were drawn from the same distribution,
  this is the probability of seeing a gap $D$ at least this large, purely
  from sampling noise

Same discipline as Tuesday's error-bar discussion: a p-value is a
statement about *what the data would look like under a specific
assumption*, not a verdict handed to you for free. Here it's vanishingly
small — the null is not a reasonable description of what we see. But
watch what happens with much smaller samples of the exact same two real
populations:

In [ ]:
rng = np.random.default_rng(20260827)
n_small = 8
sub_poor = rng.choice(alpha_poor['FeH'], n_small, replace=False)
sub_rich = rng.choice(alpha_rich['FeH'], n_small, replace=False)
ks_small, p_small = stats.ks_2samp(sub_poor, sub_rich)
print(f"with only {n_small} stars per sample: KS={ks_small:.3f}, p={p_small:.3f}")
print("Same real, genuinely-different populations. This p-value says 'not significant.'")

**The lesson:** with only 8 stars per sample, the same test that was
overwhelmingly significant on the full sample comes back saying "can't
reject the null" — a false negative, purely from bad luck in a small draw.

- a hypothesis test not rejecting $H_0$ never means "these are the same"
- it can just as easily mean "my sample was too small to tell"
- sample size and test choice both belong in your write-up, every time you
  report a p-value — exactly the kind of thing an AI assistant will not
  flag for you unprompted

## One more trap: running the same test many times

- suppose instead of one alpha cut you tried 20 different [Fe/H]/[alpha/Fe]
  binnings, looking for "a significant difference somewhere"
- at the standard $p<0.05$ threshold, you'd expect **about 1 of those 20**
  to look "significant" even if every single population were drawn from
  the exact same distribution — that's what $p=0.05$ means
- this is the multiple-comparisons problem, and it's exactly the kind of
  thing that shows up when you let an AI assistant "try a few different
  cuts and report what's significant" without telling it to correct for
  how many cuts it tried
- rule of thumb for this course: if you ran more than one test, say so in
  your write-up and say how many

## Bridge: is a single population even Gaussian?

Before you fit anything with least-squares (today's earlier segment,
Lab 01), you're implicitly assuming Gaussian residuals. A QQ-plot is the
fast visual check — same tool from the probability lecture.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(alpha_poor['FeH'], dist='norm', plot=ax)
ax.set_title('QQ-plot: alpha-poor [Fe/H] vs. a Gaussian')
plt.show()

- if the points hug the line, Gaussian is a fine working assumption
- where they peel off — usually the tails — is exactly where least-squares
  and "3-sigma means 99.7%" start lying to you
- that's the connective tissue between today's two segments: hypothesis
  tests and QQ-plots are both verification tools for assumptions you'd
  otherwise take on faith, same toolkit Tuesday introduced, applied to real
  SDSS data instead of a ten-point demo